In [ ]:
import numpy as np
import pandas as pd
import logging
from sqlalchemy import create_engine

logging.getLogger("NP.plotly").disabled = True
from neuralprophet import NeuralProphet, set_log_level

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")
warnings.simplefilter('ignore', ConvergenceWarning)

In [2]:
from dotenv import load_dotenv
from pydantic_settings import BaseSettings

load_dotenv()


class Secrets(BaseSettings):
    db_user: str
    db_password: str
    db_host: str
    db_port: str
    db_name: str
    db_table_raw_meat: str

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"


secrets = Secrets()

In [3]:
url = f'mysql+pymysql://{secrets.db_user}:{secrets.db_password}@{secrets.db_host}:{secrets.db_port}/{secrets.db_name}'

engine = create_engine(url)
with engine.connect() as conn:
    df_db = pd.read_sql(
        f'''
        SELECT *
        FROM {secrets.db_table_raw_meat}
        WHERE date >= '2020-01-01'
        AND product in ('Говядина', 'Телятина', 'Буйволятина')
        ''',
        conn,
        index_col='id'
    )

In [4]:
beef_product_types = ['азу','антрекот','аорта','балык','бедро','Бескостная','бефстроганов','бифштекс','Блочная','вымя','вырезка','глазной мускул','головы','голяшка','грудинка','грудной отруб','губы','гуляш','диафрагма','длиннейшая мышца','железа','желудки','жилка','жир','жир (сырец)','задние части','илей','калтык','карбонад','кишка','книжка','копыта','кострец','кость','легкое','лопатка','лопаточно-плечевая часть','лопаточная часть','лытка','медальон','мозги','мука мясокостная','мышца','мякоть','мясо','набор для гуляша','набор для супа','набор для тушения','набор для холодца','ноги','нос','обрезь','обрезь головная','огузок','Односортная','оковалок','окорок','отруб реберный','отруб шейный','пашина','пенис','передние части','передние четверти','печень','подбедерок','поджарка','подлопаточная часть','полутуши','почки','путовый сустав','рагу','ребро','рубец','рулька','селезенка','семенники','сердце','соединительная ткань','спинно-поясничный отруб','стейк','Субпродукты','суставы','сухожилия','сычуг','тазобедренный отруб','тонкий край','трахея','тримминг','туши','уши','фарш','филе','филей','хвосты','хрящи','черева','четверти','шейная часть','шейно-лопаточный отруб','шейный отруб','шея','шкура','шпик','шпик боковой','щека','язык']
df = df_db[df_db['product_type'].isin(beef_product_types)]

Оставим ктегории с большим количеством записей > 5000

In [5]:
group = df.groupby('product_type').count().sort_values('category', ascending=False)
categories = group[group['category'] > 5000].index.to_list()
df = df[df['product_type'].isin(categories)]
len(df)

339029

In [6]:
df.head(1)

,category,region,product,description,temperature_state,country,packaging,availability,batch,delivery_terms,...,city,date,product_type,sort,certification,federal_okrug,updated_at,week_number,file_name,hash_value
id,,,,,,,,,,,,,,,,,,,,,
3083,Мясо и мясопродукты,None,Говядина,печень 1 сорт,зам,Парагвай,коробки,склад,"от кор. до 1,5 тонн",самовывоз,...,None,2024-10-31,печень,1 сорт,None,ПФО,2024-12-25 04:48:17,44,PFO_Meat__31.10.2024.xls,9b54e050c56e3a2ea2cc260640338710bf8f31b0f0a93b...


In [7]:
df = df[['product', 'product_type', 'date', 'price']]
df.sort_values('date', ascending=True, inplace=True)
df

,product,product_type,date,price
id,,,,
1063991,Буйволятина,оковалок,2020-01-09,332.0
1064135,Говядина,лопатка,2020-01-09,322.0
1064134,Говядина,лопатка,2020-01-09,320.0
1064133,Говядина,лопатка,2020-01-09,352.0
1064132,Говядина,лопатка,2020-01-09,312.0
...,...,...,...,...
7035904,Говядина,сердце,2025-02-12,285.0
7035903,Говядина,сердце,2025-02-12,320.0
7035902,Говядина,сердце,2025-02-12,350.0


# Бейзлайн модель

In [8]:
def prepare_df_to_propheat(df):
    # считаем среднее для каждой даты по продукту и типу продукта
    g = df.copy().groupby(['date', 'product', 'product_type']).mean('price')

    # подготоавливаем датасет подходящий для propheat
    prophet_df = g.reset_index()
    prophet_df['ID'] = prophet_df['product'] + ' ' + prophet_df['product_type']
    prophet_df.drop(['product', 'product_type'], axis=1, inplace=True)
    prophet_df.rename(columns={'date':'ds', 'price':'y'}, inplace=True)

    # Удаляем категории с константной ценой
    const_g = prophet_df.groupby('ID').nunique('y')
    prophet_df.drop(prophet_df[prophet_df['ID'].isin(const_g[const_g['y'] < 3].index.to_list())].index, axis=0, inplace=True)
    return prophet_df

In [9]:
prophet_df = prepare_df_to_propheat(df)
prophet_df.head(5)

,ds,y,ID
0,2020-01-09,660.000000,Буйволятина вырезка
1,2020-01-09,306.333333,Буйволятина лопатка
2,2020-01-09,316.888889,Буйволятина огузок
3,2020-01-09,336.900000,Буйволятина оковалок
4,2020-01-09,312.272727,Буйволятина подбедерок


In [10]:
set_log_level("ERROR")

model = NeuralProphet(
    unknown_data_normalization=True,
    trend_global_local="local",
    season_global_local="local",
)
model.set_plotting_backend("plotly-static")

train, test = model.split_df(prophet_df, valid_p=0.2, local_split=True, freq="ME")

metrics = model.fit(train, freq="ME")
forecast = model.predict(test)

Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 258/258 [00:02<00:00, 95.15it/s] 


Training: |          | 0/? [01:21<?, ?it/s, v_num=155, train_loss=0.021, reg_loss=0.000, MAE=0.0916, RMSE=0.138, Loss=0.021, RegLoss=0.000]  
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 200.02it/s]


In [11]:
def plot_loss(x, y):
    mse = np.sqrt(np.mean((y - x) ** 2))
    mae = np.mean(np.abs((y - x)))
    mape = np.mean(np.abs((y - x) / y)) * 100

    print(f'mse: {mse}, mae: {mae}, mape: {round(mape, 2)}%')

plot_loss(x=forecast['yhat1'], y=forecast['y'])

mse: 134.70875405143687, mae: 50.33098191989942, mape: 11.08%


# Гипотезы

1. Обработка выбросов и обработка неправильных категорий, путем удаления 5% экстремальных значений
2. Удаление записей конкретного трейдера в говядине и баранине

Начнем с удаления выбросов

In [12]:
df

,product,product_type,date,price
id,,,,
1063991,Буйволятина,оковалок,2020-01-09,332.0
1064135,Говядина,лопатка,2020-01-09,322.0
1064134,Говядина,лопатка,2020-01-09,320.0
1064133,Говядина,лопатка,2020-01-09,352.0
1064132,Говядина,лопатка,2020-01-09,312.0
...,...,...,...,...
7035904,Говядина,сердце,2025-02-12,285.0
7035903,Говядина,сердце,2025-02-12,320.0
7035902,Говядина,сердце,2025-02-12,350.0


In [13]:
def drop_outlier(df):
    # удалем 5% данных по каждой категории в каждом продукте

    df_copy = df.copy()
    dfs = []
    for product in df_copy['product'].unique().tolist():
        for priduct_type in df_copy['product_type'].unique().tolist():
            df_ = df_copy[(df_copy['product'] == product) & (df_copy['product_type'] == priduct_type)]
            lq, rq = df_['price'].quantile(0.025), df_['price'].quantile(0.975)
            df_ = df_[(df_['price'] > lq) & (df_['price'] < rq)]

            dfs.append(df_)

    df_no_outlier = pd.concat(dfs, axis=0)
    return df_no_outlier

In [14]:
df_no_outlier = drop_outlier(df)
print(f'{len(df)} => {len(df_no_outlier)}')

339029 => 320508


In [15]:
prophet_df = prepare_df_to_propheat(df_no_outlier)

model = NeuralProphet(
    unknown_data_normalization=True,
    trend_global_local="local",
    season_global_local="local",
)
model.set_plotting_backend("plotly-static")

train, test = model.split_df(prophet_df, valid_p=0.2, local_split=True, freq="ME")

metrics = model.fit(train, freq="ME")
forecast = model.predict(test)

plot_loss(x=forecast['yhat1'], y=forecast['y'])

Epoch 50: 100%|██████████| 50/50 [00:09<00:00,  5.01it/s]   
Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 257/257 [00:02<00:00, 97.61it/s] 


Training: |          | 0/? [01:20<?, ?it/s, v_num=156, train_loss=0.0215, reg_loss=0.000, MAE=0.0955, RMSE=0.133, Loss=0.0215, RegLoss=0.000]
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 200.09it/s]
mse: 94.66729244148753, mae: 39.08251244121211, mape: 9.39%


Видно явное улучшение метрик. Теперь попробуем обработать конкретного трейдера

In [16]:
df = df_db[df_db['product_type'].isin(beef_product_types)]
group = df.groupby('product_type').count().sort_values('category', ascending=False)
categories = group[group['category'] > 5000].index.to_list()
df = df[df['product_type'].isin(categories)]
df = df[['product', 'product_type', 'date', 'company', 'price']]
df.sort_values('date', ascending=True, inplace=True)
df.head(2)

,product,product_type,date,company,price
id,,,,,
1063991,Буйволятина,оковалок,2020-01-09,Вектор СПБ,332.0
1064135,Говядина,лопатка,2020-01-09,Альфа Мит,322.0


In [17]:
def drop_traider(df):
    df_no_traider = df.drop(df[df['company'] == 'ИП Егоров А.К.'].index, axis=0)
    df_no_traider.drop('company', axis=1, inplace=True)
    return df_no_traider

df_no_traider = drop_traider(df)
df_no_traider.head()

,product,product_type,date,price
id,,,,
1063991,Буйволятина,оковалок,2020-01-09,332.0
1064135,Говядина,лопатка,2020-01-09,322.0
1064134,Говядина,лопатка,2020-01-09,320.0
1064133,Говядина,лопатка,2020-01-09,352.0
1064132,Говядина,лопатка,2020-01-09,312.0


In [18]:
prophet_df = prepare_df_to_propheat(df_no_traider)

model = NeuralProphet(
    unknown_data_normalization=True,
    trend_global_local="local",
    season_global_local="local",
)
model.set_plotting_backend("plotly-static")

train, test = model.split_df(prophet_df, valid_p=0.2, local_split=True, freq="ME")

metrics = model.fit(train, freq="ME")
forecast = model.predict(test)

plot_loss(x=forecast['yhat1'], y=forecast['y'])

Epoch 50: 100%|██████████| 50/50 [00:05<00:00,  8.80it/s]   
Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 258/258 [00:02<00:00, 99.32it/s] 


Training: |          | 0/? [01:22<?, ?it/s, v_num=157, train_loss=0.0203, reg_loss=0.000, MAE=0.0907, RMSE=0.133, Loss=0.0202, RegLoss=0.000]
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 250.02it/s]
mse: 115.73886377788084, mae: 47.29647571389807, mape: 10.96%


удаление аномального трейдера положительно повлияло на метрики. Попробуем объединить и удаление выбросов и удаление трейдера

In [19]:
df = df_db[df_db['product_type'].isin(beef_product_types)]
group = df.groupby('product_type').count().sort_values('category', ascending=False)
categories = group[group['category'] > 5000].index.to_list()
df = df[df['product_type'].isin(categories)]
df = df[['product', 'product_type', 'date', 'company', 'price']]
df.sort_values('date', ascending=True, inplace=True)
df.head(2)

,product,product_type,date,company,price
id,,,,,
1063991,Буйволятина,оковалок,2020-01-09,Вектор СПБ,332.0
1064135,Говядина,лопатка,2020-01-09,Альфа Мит,322.0


In [20]:
df_no_traider = drop_traider(df)
df_no_outlier_traider = drop_outlier(df_no_traider)

In [21]:
prophet_df = prepare_df_to_propheat(df_no_outlier_traider)

model = NeuralProphet(
    unknown_data_normalization=True,
    trend_global_local="local",
    season_global_local="local",
)
model.set_plotting_backend("plotly-static")

train, test = model.split_df(prophet_df, valid_p=0.2, local_split=True, freq="ME")

metrics = model.fit(train, freq="ME")
forecast = model.predict(test)

plot_loss(x=forecast['yhat1'], y=forecast['y'])

Epoch 50: 100%|██████████| 50/50 [00:11<00:00,  4.36it/s]   
Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 257/257 [00:02<00:00, 95.97it/s] 


Training: |          | 0/? [01:20<?, ?it/s, v_num=158, train_loss=0.0206, reg_loss=0.000, MAE=0.0931, RMSE=0.130, Loss=0.0206, RegLoss=0.000]
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 252.15it/s]
mse: 94.04719990612593, mae: 38.59943854465225, mape: 9.43%


Удаление аномального трейдера и удаление выбросов позитивно влияют на метрики

## Добавление макроэкономических показателей в качестве глобального тренда

In [22]:
from neuralprophet import set_random_seed
import torch
import numpy as np
import random
import os

set_random_seed(0)
set_log_level("ERROR")

def reset_random_state(seed): 
    os.environ['PYTHONHASHSEED']=str(seed) 
    torch.manual_seed(seed) 
    np.random.seed(seed) 
    random.seed(seed)

reset_random_state(0)    

In [ ]:
engine = create_engine(url)
with engine.connect() as conn:
    df_db = pd.read_sql(
        f'''
        SELECT *
        FROM {secrets.db_table_raw_meat}
        WHERE date >= '2020-01-01'
        AND product in ('Говядина', 'Телятина', 'Буйволятина')
        ''',
        conn,
        index_col='id'
    )

In [24]:
date_range_df = pd.DataFrame(pd.date_range(start='01-01-2020', end='22-01-2025', freq='1d'), columns=['ds'])

# добавление макро показателей
macro_df = pd.read_csv('macroeconomic_indicators.csv', encoding='utf-8')

cols = ['ВВП','Индекс пром.производства','Цены производителей пром.товаров','Производство пищевых продуктов','Инфляция','Индекс реальной зарплаты']
for col in cols:
    macro_df[col] = macro_df[col].apply(lambda x: float(x[:-1].replace(',', '.')))

macro_df['ds'] = pd.to_datetime(macro_df['ds'], yearfirst=False, dayfirst=True)
macro_df = pd.merge(date_range_df, macro_df, on='ds', how='left').ffill()

# добавление курса рубля к доллару
df_rub = pd.read_csv('rub_dollar_course.csv', encoding='windows-1251')
df_rub['data'] = pd.to_datetime(df_rub['data'])
df_rub = df_rub.drop(['nominal', 'cdx'], axis=1)
df_rub.rename(columns={'data':'ds'}, inplace=True)
df_rub = pd.merge(date_range_df, df_rub, on='ds', how='left').ffill().bfill()
df_rub['curs'] = df_rub['curs'].apply(lambda x:float(x.replace(',', '.')))

In [25]:
df_data = df_db.copy()

beef_product_types = ['азу','антрекот','аорта','балык','бедро','Бескостная','бефстроганов','бифштекс','Блочная','вымя','вырезка','глазной мускул','головы','голяшка','грудинка','грудной отруб','губы','гуляш','диафрагма','длиннейшая мышца','железа','желудки','жилка','жир','жир (сырец)','задние части','илей','калтык','карбонад','кишка','книжка','копыта','кострец','кость','легкое','лопатка','лопаточно-плечевая часть','лопаточная часть','лытка','медальон','мозги','мука мясокостная','мышца','мякоть','мясо','набор для гуляша','набор для супа','набор для тушения','набор для холодца','ноги','нос','обрезь','обрезь головная','огузок','Односортная','оковалок','окорок','отруб реберный','отруб шейный','пашина','пенис','передние части','передние четверти','печень','подбедерок','поджарка','подлопаточная часть','полутуши','почки','путовый сустав','рагу','ребро','рубец','рулька','селезенка','семенники','сердце','соединительная ткань','спинно-поясничный отруб','стейк','Субпродукты','суставы','сухожилия','сычуг','тазобедренный отруб','тонкий край','трахея','тримминг','туши','уши','фарш','филе','филей','хвосты','хрящи','черева','четверти','шейная часть','шейно-лопаточный отруб','шейный отруб','шея','шкура','шпик','шпик боковой','щека','язык']
df_data = df_data[df_data['product_type'].isin(beef_product_types)]

group = df_data.groupby('product_type').count().sort_values('category', ascending=False)
categories = group[group['category'] > 5000].index.to_list()
df_data = df_data[df_data['product_type'].isin(categories)]
df_data = df_data[['product', 'product_type', 'date', 'price']]
df_data.sort_values('date', ascending=True, inplace=True)

In [26]:
prophet_df = prepare_df_to_propheat(df_data)

In [27]:
nd = len(date_range_df)
nu = len(prophet_df['ID'].unique())

df_final = pd.DataFrame(
    {
        'ID': prophet_df['ID'].unique().tolist() * nd,
        'ds': np.repeat(date_range_df, nu)
    }
)
df_final = df_final.merge(prophet_df, how='left')

In [28]:
df = pd.merge(df_final, macro_df, on='ds', how='left')
df = pd.merge(df, df_rub, on='ds', how='left')

In [29]:
df.rename(columns={'price':'y'}, inplace=True)

In [30]:
len(df)

86903

In [31]:
for cat in df['ID'].unique().tolist():
    df_ = df[df['ID'] == cat].interpolate('linear', limit_direction='both')
    df.loc[df_.index, 'y'] = df_['y']

In [32]:
model = NeuralProphet(
    drop_missing=True,
    unknown_data_normalization=True,
    # тип трендов локальный / глобальный
    trend_global_local="local",
    season_global_local="local",
    # сезонные тренды
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,

    n_forecasts=90,
    n_lags=90,  # Autogression
)
model.set_plotting_backend("plotly-static")

model.add_lagged_regressor('ВВП', n_lags=90)
model.add_lagged_regressor('Индекс пром.производства', n_lags=90)
model.add_lagged_regressor('Цены производителей пром.товаров', n_lags=90)
model.add_lagged_regressor('Производство пищевых продуктов', n_lags=90)
model.add_lagged_regressor('Инфляция', n_lags=90)
model.add_lagged_regressor('Индекс реальной зарплаты', n_lags=90)
model.add_lagged_regressor('curs', n_lags=90)

df_train, df_val = model.split_df(df, valid_p=0.3, local_split=False, freq="D")

print("Dataset size:", len(prophet_df))
print("Train dataset size:", len(df_train))
print("Validation dataset size:", len(df_val))

reset_random_state(0)    
metrics = model.fit(df_train, validation_df=df_val, freq="D")

Dataset size: 26291
Train dataset size: 60442
Validation dataset size: 30691
Epoch 50: 100%|██████████| 50/50 [00:29<00:00,  1.70it/s]   
Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 269/269 [00:05<00:00, 52.32it/s]


Training: |          | 0/? [02:33<?, ?it/s, v_num=159, MAE_val=0.373, RMSE_val=0.762, Loss_val=0.275, RegLoss_val=0.000, train_loss=0.0275, reg_loss=0.000, MAE=0.104, RMSE=0.158, Loss=0.0274, RegLoss=0.000]


In [33]:
df_to_preds = model.make_future_dataframe(df_train, n_historic_predictions=False, periods=90)
forecast = model.predict(df_to_preds)

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 62.50it/s] 


In [34]:
# оставляем значения без предсказания
forecast = forecast[forecast['y'].isna()]
# обрабатываем предсказанные значения
yhats = ['yhat' + str(i) for i in range(1, 91)]
forecast['yhat'] = forecast[yhats].max(axis=1)
# соединяем предсказания с тестовыми данными
forecast = pd.merge(forecast, df_val[['ID', 'ds', 'y']], how='left', on=['ID', 'ds'])

# считаем метрики
plot_loss(x=forecast['yhat'], y=forecast['y_y'])

mse: 109.4010591556555, mae: 60.366084578704395, mape: 11.89%


Макроэкономические показатели положительно влияют на метрики